# Snowflake Database Exploratory Data Analysis (EDA)

This notebook provides comprehensive exploratory data analysis for Snowflake databases. It allows you to:
1. List all available databases
2. Explore tables within selected databases
3. Analyze column structures and data types
4. Examine sample data and basic statistics
5. Visualize database relationships

The notebook operates automatically without requiring human interaction after initial configuration.

In [1]:
# Install required packages
%pip install -q snowflake-connector-python pandas matplotlib seaborn plotly python-dotenv


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import pandas as pd
import snowflake.connector
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML, Markdown
from datetime import datetime
from dotenv import load_dotenv

# Set up color schemes and styling
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")

# Load environment variables if any
load_dotenv()

# Configure pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 100)

# Helper function for better DataFrame display
def pretty_display(df, title=None, max_rows=20):
    """Display a DataFrame with title and styling."""
    if title:
        display(Markdown(f"### {title}"))
    
    # Create styled HTML representation with row alternating colors
    styled_df = df.head(max_rows).style.set_properties(**{
        'background-color': '#f5f5f5',
        'border-color': '#888888',
        'border-style': 'solid',
        'border-width': '1px',
        'text-align': 'left'
    }).set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#4CAF50'), 
                                     ('color', 'white'),
                                     ('font-weight', 'bold'),
                                     ('text-align', 'left'),
                                     ('padding', '8px')]},
        {'selector': 'td', 'props': [('padding', '8px')]}
    ]).hide(axis="index")
    
    # Add alternating row colors
    styled_df = styled_df.set_table_styles([
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f2f2f2')]},
        {'selector': 'tr:hover', 'props': [('background-color', '#ddd')]}
    ], overwrite=False)
    
    display(styled_df)
    
    # Display row count information
    if len(df) > max_rows:
        display(Markdown(f"*Showing {max_rows} of {len(df)} rows*"))

In [3]:
# Snowflake connection manager
class SnowflakeExplorer:
    """Manages connection to Snowflake and provides exploration capabilities."""
    
    def __init__(self, credentials_path=None):
        """
        Initialize the Snowflake Explorer.
        
        Args:
            credentials_path: Path to Snowflake credentials JSON file
        """
        self.credentials_path = credentials_path or os.path.join(
            os.path.abspath('/Users/A200303816/Documents/LLM and Agents/ProjectWork/Spider2'), 
            "methods", 
            "spider-agent-snow", 
            "snowflake_credential.json"
        )
        self.conn = None
        self.cursor = None
        self.credentials = None
        self.current_database = None
        self.current_schema = None
        self.databases_df = None
        
        # Metrics tracking
        self.metrics = {
            "database_count": 0,
            "schema_count": 0,
            "table_count": 0,
            "column_count": 0,
            "data_type_distribution": {},
            "last_updated": None
        }
        
        print("Snowflake Explorer initialized")
    
    def load_credentials(self):
        """Load Snowflake credentials from JSON file."""
        try:
            with open(self.credentials_path, 'r') as file:
                self.credentials = json.load(file)
            print("✅ Credentials loaded successfully")
            return True
        except Exception as e:
            print(f"❌ Failed to load credentials: {e}")
            return False
    
    def connect(self):
        """Establish connection to Snowflake."""
        if not self.credentials:
            if not self.load_credentials():
                return False
        
        try:
            self.conn = snowflake.connector.connect(
                user=self.credentials['user'],
                password=self.credentials['password'],
                account=self.credentials['account']
            )
            self.cursor = self.conn.cursor()
            
            # Test connection
            self.cursor.execute("SELECT current_version()")
            version = self.cursor.fetchone()[0]
            print(f"✅ Successfully connected to Snowflake (Version: {version})")
            return True
        except Exception as e:
            print(f"❌ Failed to connect to Snowflake: {e}")
            return False
    
    def get_databases(self):
        """Get list of all accessible databases."""
        if not self.cursor:
            print("No active connection to Snowflake")
            return None
        
        try:
            self.cursor.execute("SHOW DATABASES")
            results = self.cursor.fetchall()
            columns = [desc[0] for desc in self.cursor.description]
            self.databases_df = pd.DataFrame(results, columns=columns)
            
            # Update metrics
            self.metrics["database_count"] = len(self.databases_df)
            self.metrics["last_updated"] = datetime.now().isoformat()
            
            return self.databases_df
        except Exception as e:
            print(f"Failed to list databases: {e}")
            return None
    
    def use_database(self, database_name):
        """Select a database to use."""
        if not self.cursor:
            print("No active connection to Snowflake")
            return False
        
        try:
            self.cursor.execute(f"USE DATABASE {database_name}")
            self.current_database = database_name
            print(f"Now using database: {database_name}")
            return True
        except Exception as e:
            print(f"Failed to use database {database_name}: {e}")
            return False
    
    def get_schemas(self, database_name=None):
        """Get list of all schemas in a database."""
        if database_name and database_name != self.current_database:
            if not self.use_database(database_name):
                return None
        
        if not self.cursor or not self.current_database:
            print("No active database selected")
            return None
        
        try:
            self.cursor.execute("SHOW SCHEMAS")
            results = self.cursor.fetchall()
            columns = [desc[0] for desc in self.cursor.description]
            schemas_df = pd.DataFrame(results, columns=columns)
            
            # Update metrics
            self.metrics["schema_count"] = len(schemas_df)
            
            return schemas_df
        except Exception as e:
            print(f"Failed to list schemas: {e}")
            return None
    
    def use_schema(self, schema_name):
        """Select a schema to use."""
        if not self.cursor or not self.current_database:
            print("No active database selected")
            return False
        
        try:
            self.cursor.execute(f"USE SCHEMA {schema_name}")
            self.current_schema = schema_name
            print(f"Now using schema: {schema_name}")
            return True
        except Exception as e:
            print(f"Failed to use schema {schema_name}: {e}")
            return False
    
    def get_tables(self, schema_name=None):
        """Get list of all tables in a schema."""
        if schema_name and schema_name != self.current_schema:
            if not self.use_schema(schema_name):
                return None
        
        if not self.cursor or not self.current_schema:
            print("No active schema selected")
            return None
        
        try:
            self.cursor.execute("SHOW TABLES")
            results = self.cursor.fetchall()
            columns = [desc[0] for desc in self.cursor.description]
            tables_df = pd.DataFrame(results, columns=columns)
            
            # Update metrics
            self.metrics["table_count"] += len(tables_df)
            
            return tables_df
        except Exception as e:
            print(f"Failed to list tables: {e}")
            return None
    
    def get_table_schema(self, table_name):
        """Get schema information for a specific table."""
        if not self.cursor or not self.current_schema:
            print("No active schema selected")
            return None
        
        try:
            self.cursor.execute(f"DESCRIBE TABLE {table_name}")
            results = self.cursor.fetchall()
            columns = [desc[0] for desc in self.cursor.description]
            table_schema_df = pd.DataFrame(results, columns=columns)
            
            # Update metrics
            self.metrics["column_count"] += len(table_schema_df)
            
            # Update data type distribution
            data_types = table_schema_df['type'].value_counts().to_dict()
            for dtype, count in data_types.items():
                if dtype in self.metrics["data_type_distribution"]:
                    self.metrics["data_type_distribution"][dtype] += count
                else:
                    self.metrics["data_type_distribution"][dtype] = count
            
            return table_schema_df
        except Exception as e:
            print(f"Failed to describe table {table_name}: {e}")
            return None
    
    def get_table_sample(self, table_name, limit=10):
        """Get sample rows from a table."""
        if not self.cursor or not self.current_schema:
            print("No active schema selected")
            return None
        
        try:
            # Get column names first for better display
            self.cursor.execute(f"SELECT * FROM {table_name} LIMIT 0")
            columns = [desc[0] for desc in self.cursor.description]
            
            # Now get actual data
            self.cursor.execute(f"SELECT * FROM {table_name} LIMIT {limit}")
            results = self.cursor.fetchall()
            sample_df = pd.DataFrame(results, columns=columns)
            
            return sample_df
        except Exception as e:
            print(f"Failed to sample table {table_name}: {e}")
            return None
    
    def get_row_count(self, table_name):
        """Get the row count for a table."""
        if not self.cursor or not self.current_schema:
            print("No active schema selected")
            return None
        
        try:
            self.cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
            count = self.cursor.fetchone()[0]
            return count
        except Exception as e:
            print(f"Failed to get row count for {table_name}: {e}")
            return None
    
    def get_table_statistics(self, table_name):
        """Get basic statistics for columns in a table."""
        if not self.cursor or not self.current_schema:
            print("No active schema selected")
            return None
        
        # Get table schema first to identify numeric columns
        schema_df = self.get_table_schema(table_name)
        if schema_df is None:
            return None
        
        # Find numeric columns
        numeric_types = ['NUMBER', 'FLOAT', 'INTEGER', 'INT', 'BIGINT', 'DECIMAL', 'NUMERIC']
        numeric_columns = schema_df[schema_df['type'].str.upper().str.contains('|'.join(numeric_types), 
                                                                            case=False, 
                                                                            regex=True)]['name'].tolist()
        
        if not numeric_columns:
            return pd.DataFrame(columns=['column_name', 'min', 'max', 'avg', 'sum'])
        
        stats = []
        for col in numeric_columns:
            try:
                self.cursor.execute(f"""
                SELECT 
                    '{col}' as column_name,
                    MIN({col}) as min_val,
                    MAX({col}) as max_val,
                    AVG({col}) as avg_val,
                    SUM({col}) as sum_val
                FROM {table_name}
                """)
                row = self.cursor.fetchone()
                stats.append({
                    'column_name': col,
                    'min': row[1],
                    'max': row[2],
                    'avg': row[3],
                    'sum': row[4]
                })
            except Exception as e:
                print(f"Could not get statistics for column {col}: {e}")
        
        return pd.DataFrame(stats)
    
    def explore_database(self, database_name, max_tables=5, max_columns=10):
        """Comprehensive exploration of a database."""
        if not self.use_database(database_name):
            return None
        
        print(f"\n🔍 Exploring database: {database_name}")
        results = {
            "database_name": database_name,
            "schemas": [],
            "tables_count": 0,
            "columns_count": 0
        }
        
        # Get schemas
        schemas_df = self.get_schemas()
        if schemas_df is None or len(schemas_df) == 0:
            print(f"No schemas found in database {database_name}")
            return results
        
        pretty_display(schemas_df, f"Schemas in {database_name}")
        
        # Limit number of schemas to explore
        schemas_to_explore = schemas_df['name'].head(3).tolist()
        
        # Explore each schema
        for schema_name in schemas_to_explore:
            if not self.use_schema(schema_name):
                continue
                
            schema_result = {
                "schema_name": schema_name,
                "tables": []
            }
            
            # Get tables
            tables_df = self.get_tables()
            if tables_df is None or len(tables_df) == 0:
                print(f"No tables found in schema {schema_name}")
                continue
                
            pretty_display(tables_df, f"Tables in {schema_name} schema")
            
            # Limit the number of tables to explore
            tables_to_explore = tables_df['name'].head(max_tables).tolist()
            
            # Explore each table
            for table_name in tables_to_explore:
                table_result = {
                    "table_name": table_name,
                    "column_count": 0,
                    "row_count": 0
                }
                
                # Get table schema
                table_schema = self.get_table_schema(table_name)
                if table_schema is not None:
                    pretty_display(table_schema.head(max_columns), f"Columns in {table_name} table")
                    table_result["column_count"] = len(table_schema)
                    
                # Get row count
                row_count = self.get_row_count(table_name)
                if row_count is not None:
                    table_result["row_count"] = row_count
                    print(f"Total rows in {table_name}: {row_count:,}")
                
                # Get sample data
                sample_df = self.get_table_sample(table_name, limit=5)
                if sample_df is not None and not sample_df.empty:
                    pretty_display(sample_df, f"Sample data from {table_name}")
                
                schema_result["tables"].append(table_result)
                results["columns_count"] += table_result["column_count"]
            
            results["tables_count"] += len(schema_result["tables"])
            results["schemas"].append(schema_result)
        
        return results
    
    def get_all_structures(self, max_databases=3):
        """Get complete structure of Snowflake account."""
        if not self.cursor:
            if not self.connect():
                return None
        
        # Get all databases
        databases_df = self.get_databases()
        if databases_df is None:
            return None
        
        pretty_display(databases_df, "All Available Databases")
        
        # Initialize results structure
        account_structure = {
            "databases": [],
            "total_schemas": 0,
            "total_tables": 0,
            "total_columns": 0
        }
        
        # Explore a subset of databases
        databases_to_explore = databases_df['name'].head(max_databases).tolist()
        
        for database_name in databases_to_explore:
            db_result = self.explore_database(database_name)
            if db_result:
                account_structure["databases"].append(db_result)
                account_structure["total_schemas"] += len(db_result["schemas"])
                account_structure["total_tables"] += db_result["tables_count"]
                account_structure["total_columns"] += db_result["columns_count"]
        
        return account_structure
    
    def close(self):
        """Close Snowflake connection."""
        if self.cursor:
            self.cursor.close()
        if self.conn:
            self.conn.close()
            print("Snowflake connection closed")

In [4]:
# Initialize and connect to Snowflake
explorer = SnowflakeExplorer()
connection_successful = explorer.connect()

if not connection_successful:
    print("\n❌ Could not connect to Snowflake. Please check your credentials.")
else:
    print("\n✅ Connected to Snowflake successfully! Now exploring databases...")

Snowflake Explorer initialized
✅ Credentials loaded successfully


✅ Successfully connected to Snowflake (Version: 9.12.1)

✅ Connected to Snowflake successfully! Now exploring databases...


In [5]:
# EDA functions for visualization and analysis
class SnowflakeEDA:
    """Provides EDA visualizations and analyses for Snowflake data."""
    
    def __init__(self, explorer):
        """Initialize with a SnowflakeExplorer instance."""
        self.explorer = explorer
    
    def plot_database_sizes(self):
        """Plot database sizes."""
        if self.explorer.databases_df is None:
            self.explorer.get_databases()
            
        if self.explorer.databases_df is None:
            print("No database information available")
            return
        
        # Extract relevant columns
        if 'bytes' in self.explorer.databases_df.columns:
            # Create size in MB
            df = self.explorer.databases_df.copy()
            df['size_mb'] = df['bytes'] / (1024 * 1024)
            
            # Create bar plot
            fig = px.bar(
                df.sort_values('size_mb', ascending=False).head(10),
                x='name',
                y='size_mb',
                title='Top 10 Databases by Size (MB)',
                labels={'name': 'Database', 'size_mb': 'Size (MB)'},
                color='size_mb',
                color_continuous_scale='Viridis'
            )
            
            fig.update_layout(
                xaxis_title='Database Name',
                yaxis_title='Size (MB)',
                template='plotly_white'
            )
            
            fig.show()
        else:
            print("Size information not available in database listing")
    
    def plot_database_metrics(self, account_structure):
        """Plot metrics about databases, schemas, and tables."""
        if not account_structure:
            print("No account structure data available")
            return
        
        # Extract data for plotting
        db_names = [db["database_name"] for db in account_structure["databases"]]
        schemas_counts = [len(db["schemas"]) for db in account_structure["databases"]]
        tables_counts = [db["tables_count"] for db in account_structure["databases"]]
        columns_counts = [db["columns_count"] for db in account_structure["databases"]]
        
        # Create DataFrame for plotting
        metrics_df = pd.DataFrame({
            'Database': db_names,
            'Schemas': schemas_counts,
            'Tables': tables_counts,
            'Columns': columns_counts
        })
        
        # Create bar plots
        fig = make_subplots(
            rows=3, 
            cols=1,
            subplot_titles=('Schemas per Database', 'Tables per Database', 'Columns per Database'),
            vertical_spacing=0.1
        )
        
        # Schemas plot
        fig.add_trace(
            go.Bar(x=metrics_df['Database'], y=metrics_df['Schemas'], name='Schemas', marker_color='royalblue'),
            row=1, col=1
        )
        
        # Tables plot
        fig.add_trace(
            go.Bar(x=metrics_df['Database'], y=metrics_df['Tables'], name='Tables', marker_color='darkorange'),
            row=2, col=1
        )
        
        # Columns plot
        fig.add_trace(
            go.Bar(x=metrics_df['Database'], y=metrics_df['Columns'], name='Columns', marker_color='green'),
            row=3, col=1
        )
        
        fig.update_layout(
            height=900,
            width=800,
            title_text='Database Structure Metrics',
            showlegend=False,
            template='plotly_white'
        )
        
        fig.show()
        
        # Also display as a table
        pretty_display(metrics_df, "Database Structure Summary")
    
    def plot_data_type_distribution(self):
        """Plot distribution of data types across explored tables."""
        if not self.explorer.metrics["data_type_distribution"]:
            print("No data type information available")
            return
        
        # Create DataFrame from data type distribution
        dtype_df = pd.DataFrame([
            {'Data Type': dtype, 'Count': count}
            for dtype, count in self.explorer.metrics["data_type_distribution"].items()
        ])
        
        # Sort by count
        dtype_df = dtype_df.sort_values('Count', ascending=False)
        
        # Create pie chart
        fig = px.pie(
            dtype_df,
            values='Count',
            names='Data Type',
            title='Distribution of Column Data Types',
            color_discrete_sequence=px.colors.qualitative.Pastel
        )
        
        fig.update_layout(
            legend_title='Data Type',
            template='plotly_white'
        )
        
        fig.show()
        
        # Also display as a table
        pretty_display(dtype_df, "Data Type Distribution")
    
    def visualize_table_relationships(self, database_name, schema_name):
        """Attempt to visualize table relationships based on column names."""
        if not self.explorer.use_database(database_name) or not self.explorer.use_schema(schema_name):
            return
        
        tables_df = self.explorer.get_tables()
        if tables_df is None or len(tables_df) == 0:
            print(f"No tables found in {database_name}.{schema_name}")
            return
        
        # Get table information with column details
        table_info = {}
        for table_name in tables_df['name'].head(10).tolist():  # Limit to 10 tables
            schema = self.explorer.get_table_schema(table_name)
            if schema is not None:
                table_info[table_name] = {
                    'columns': schema['name'].tolist(),
                    'column_types': dict(zip(schema['name'], schema['type'])),
                    'primary_keys': schema[schema['null?'] == 'N']['name'].tolist(),
                }
        
        # Function to find potential relationships
        def find_relationships(table_info):
            relationships = []
            
            for table1, info1 in table_info.items():
                for table2, info2 in table_info.items():
                    if table1 != table2:
                        # Look for columns with same name (potential foreign keys)
                        for col in info1['columns']:
                            if col in info2['columns']:
                                # Check if it's a primary key in one of the tables
                                is_pk1 = col in info1.get('primary_keys', [])
                                is_pk2 = col in info2.get('primary_keys', [])
                                
                                if is_pk1 or is_pk2:
                                    relationships.append({
                                        'from_table': table1,
                                        'to_table': table2,
                                        'column': col,
                                        'is_pk1': is_pk1,
                                        'is_pk2': is_pk2
                                    })
            
            return relationships
        
        relationships = find_relationships(table_info)
        
        # Create network graph
        if relationships:
            from plotly.subplots import make_subplots
            import networkx as nx
            
            # Create graph
            G = nx.DiGraph()
            
            # Add nodes (tables)
            for table in table_info.keys():
                G.add_node(table, size=len(table_info[table]['columns']))
            
            # Add edges (relationships)
            for rel in relationships:
                G.add_edge(rel['from_table'], rel['to_table'], 
                           label=rel['column'], 
                           width=2 if rel['is_pk1'] or rel['is_pk2'] else 1)
            
            # Use networkx spring layout
            pos = nx.spring_layout(G, seed=42)
            
            # Create traces
            edge_trace = go.Scatter(
                x=[],
                y=[],
                line=dict(width=0.5, color='#888'),
                hoverinfo='none',
                mode='lines')
            
            for edge in G.edges():
                x0, y0 = pos[edge[0]]
                x1, y1 = pos[edge[1]]
                edge_trace['x'] += tuple([x0, x1, None])
                edge_trace['y'] += tuple([y0, y1, None])
            
            node_trace = go.Scatter(
                x=[],
                y=[],
                text=[],
                mode='markers+text',
                textposition='top center',
                hoverinfo='text',
                marker=dict(
                    showscale=True,
                    colorscale='YlGnBu',
                    reversescale=True,
                    color=[],
                    size=20,
                    colorbar=dict(
                        thickness=15,
                        title='Node Connections',
                        xanchor='left',
                        titleside='right'
                    ),
                    line=dict(width=2)))
            
            for node in G.nodes():
                x, y = pos[node]
                node_trace['x'] += tuple([x])
                node_trace['y'] += tuple([y])
                node_trace['text'] += tuple([node])
                node_trace['marker']['color'] += tuple([len(G.edges(node))])
                node_trace['marker']['size'] += tuple([15 + G.nodes[node]['size']])
            
            # Create figure
            fig = go.Figure(data=[edge_trace, node_trace],
                            layout=go.Layout(
                                title=f'Table Relationships in {database_name}.{schema_name}',
                                titlefont=dict(size=16),
                                showlegend=False,
                                hovermode='closest',
                                margin=dict(b=20,l=5,r=5,t=40),
                                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
                           )
            
            fig.show()
            
            # Also display relationships as a table
            rel_df = pd.DataFrame(relationships)
            pretty_display(rel_df, "Potential Table Relationships")
        else:
            print("No clear relationships identified between tables")

In [ ]:
# Run the EDA
if connection_successful:
    try:
        print("\n🔍 Starting Exploratory Data Analysis...")
        
        # Automatically explore the Snowflake account
        account_structure = explorer.get_all_structures(max_databases=3)
        
        # Create EDA helper
        eda = SnowflakeEDA(explorer)
        
        # Visualize database metrics
        if account_structure:
            from plotly.subplots import make_subplots
            import plotly.graph_objects as go
            
            print("\n📊 Generating visualizations of database structures...")
            eda.plot_database_metrics(account_structure)
            
            # Visualize data types
            print("\n📊 Analyzing data type distributions...")
            eda.plot_data_type_distribution()
            
            # Try to visualize relationships in one database-schema
            if account_structure["databases"]:
                first_db = account_structure["databases"][0]
                if first_db["schemas"]:
                    first_schema = first_db["schemas"][0]["schema_name"]
                    print(f"\n🔗 Analyzing potential relationships in {first_db['database_name']}.{first_schema}...")
                    eda.visualize_table_relationships(first_db["database_name"], first_schema)
        
        # Create summary report
        print("\n📋 EDA Summary:")
        print(f"- Total databases explored: {len(account_structure['databases'])}")
        print(f"- Total schemas: {account_structure['total_schemas']}")
        print(f"- Total tables: {account_structure['total_tables']}")
        print(f"- Total columns: {account_structure['total_columns']}")
        
        # Close the connection
        explorer.close()
        
    except Exception as e:
        print(f"❌ Error during exploration: {e}")
        explorer.close()

In [7]:
# Function to automatically explore a specific database in detail
def deep_dive_database(database_name):
    """
    Perform a deep dive analysis of a specific database.
    
    Args:
        database_name: Name of the database to analyze
    """
    print(f"\n🔍 Deep diving into database: {database_name}")
    
    # Re-connect to Snowflake
    explorer = SnowflakeExplorer()
    if not explorer.connect():
        print("Failed to connect to Snowflake")
        return
    
    # Use the specified database
    if not explorer.use_database(database_name):
        print(f"Failed to use database {database_name}")
        return
    
    # Get schemas in the database
    schemas_df = explorer.get_schemas()
    if schemas_df is None or len(schemas_df) == 0:
        print(f"No schemas found in database {database_name}")
        return
    
    pretty_display(schemas_df, f"Schemas in {database_name}")
    
    # Process each schema
    for schema_name in schemas_df['name'].tolist():
        if not explorer.use_schema(schema_name):
            continue
            
        print(f"\n📊 Analyzing schema: {schema_name}")
        
        # Get tables in the schema
        tables_df = explorer.get_tables()
        if tables_df is None or len(tables_df) == 0:
            print(f"No tables found in schema {schema_name}")
            continue
            
        pretty_display(tables_df, f"Tables in {schema_name}")
        
        # Count total rows in the schema
        total_rows = 0
        table_sizes = []
        
        for table_name in tables_df['name'].tolist():
            row_count = explorer.get_row_count(table_name)
            if row_count is not None:
                total_rows += row_count
                table_sizes.append({
                    'table_name': table_name,
                    'row_count': row_count
                })
        
        # Display table sizes
        if table_sizes:
            sizes_df = pd.DataFrame(table_sizes).sort_values('row_count', ascending=False)
            pretty_display(sizes_df, f"Table Sizes in {schema_name}")
            
            # Create bar chart of table sizes
            plt.figure(figsize=(12, 6))
            sns.barplot(x='table_name', y='row_count', data=sizes_df.head(15))
            plt.title(f'Top 15 Tables by Row Count in {schema_name}')
            plt.xlabel('Table Name')
            plt.ylabel('Number of Rows')
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()
        
        print(f"Total rows in schema {schema_name}: {total_rows:,}")
        
        # Analyze column data types in the schema
        column_types = []
        for table_name in tables_df['name'].head(20).tolist():  # Limit to 20 tables
            schema = explorer.get_table_schema(table_name)
            if schema is not None:
                for _, row in schema.iterrows():
                    column_types.append({
                        'table_name': table_name,
                        'column_name': row['name'],
                        'data_type': row['type']
                    })
        
        if column_types:
            types_df = pd.DataFrame(column_types)
            
            # Count data types
            type_counts = types_df['data_type'].value_counts().reset_index()
            type_counts.columns = ['data_type', 'count']
            
            pretty_display(type_counts, f"Data Type Distribution in {schema_name}")
            
            # Create pie chart
            plt.figure(figsize=(10, 6))
            plt.pie(type_counts['count'], labels=type_counts['data_type'], autopct='%1.1f%%')
            plt.title(f'Data Type Distribution in {schema_name}')
            plt.axis('equal')
            plt.tight_layout()
            plt.show()
    
    # Close the connection
    explorer.close()
    print(f"\n✅ Deep dive into {database_name} completed")

# Uncomment to run a deep dive on a specific database
# deep_dive_database("ADVENTUREWORKS")

In [8]:
# Analyze relations between tables (foreign keys)
def analyze_foreign_keys(database_name, schema_name):
    """
    Analyze potential foreign key relationships between tables.
    
    Args:
        database_name: Name of the database to analyze
        schema_name: Name of the schema to analyze
    """
    print(f"\n🔍 Analyzing relationships in {database_name}.{schema_name}")
    
    # Connect to Snowflake
    explorer = SnowflakeExplorer()
    if not explorer.connect():
        print("Failed to connect to Snowflake")
        return
    
    # Use the specified database and schema
    if not explorer.use_database(database_name) or not explorer.use_schema(schema_name):
        print(f"Failed to use {database_name}.{schema_name}")
        return
    
    # Get tables
    tables_df = explorer.get_tables()
    if tables_df is None or len(tables_df) == 0:
        print(f"No tables found in {database_name}.{schema_name}")
        return
    
    # Get table information with column details
    table_info = {}
    for table_name in tables_df['name'].tolist():
        schema = explorer.get_table_schema(table_name)
        if schema is not None:
            table_info[table_name] = {
                'columns': schema['name'].tolist(),
                'column_types': dict(zip(schema['name'], schema['type'])),
                'primary_keys': schema[schema['null?'] == 'N']['name'].tolist(),
            }
    
    # Find potential relationships based on column name patterns
    relationships = []
    for table1, info1 in table_info.items():
        for table2, info2 in table_info.items():
            if table1 != table2:
                for col1 in info1['columns']:
                    # Check for exact matches
                    if col1 in info2['columns']:
                        relationships.append({
                            'from_table': table1,
                            'to_table': table2,
                            'from_column': col1,
                            'to_column': col1,
                            'match_type': 'exact'
                        })
                    # Check for key pattern matches (e.g., CustomerId in one table, Customer_Id in another)
                    else:
                        # Normalize column names by removing underscores and lowercasing
                        col1_norm = col1.replace('_', '').lower()
                        for col2 in info2['columns']:
                            col2_norm = col2.replace('_', '').lower()
                            # Check for common foreign key pattern (table name + Id)
                            if col2_norm == table1.lower() + 'id' or col1_norm == table2.lower() + 'id':
                                relationships.append({
                                    'from_table': table1,
                                    'to_table': table2,
                                    'from_column': col1,
                                    'to_column': col2,
                                    'match_type': 'pattern'
                                })
    
    # Display relationships
    if relationships:
        rel_df = pd.DataFrame(relationships)
        pretty_display(rel_df, "Potential Foreign Key Relationships")
        
        # Create graph visualization of relationships
        try:
            import networkx as nx
            
            # Create a directed graph
            G = nx.DiGraph()
            
            # Add nodes (tables)
            for table in table_info.keys():
                G.add_node(table)
            
            # Add edges (relationships)
            for rel in relationships:
                if rel['match_type'] == 'exact':  # Only use exact matches for clarity
                    G.add_edge(rel['from_table'], rel['to_table'], 
                              label=f"{rel['from_column']} -> {rel['to_column']}")
            
            # Plot the graph
            plt.figure(figsize=(12, 12))
            pos = nx.spring_layout(G, seed=42)
            nx.draw_networkx_nodes(G, pos, node_size=700, node_color='lightblue')
            nx.draw_networkx_edges(G, pos, width=1.0, arrowsize=20)
            nx.draw_networkx_labels(G, pos, font_size=10)
            
            plt.title(f"Table Relationships in {database_name}.{schema_name}")
            plt.axis('off')
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Error creating graph visualization: {e}")
    else:
        print("No clear relationships identified between tables")
    
    # Close the connection
    explorer.close()

# Uncomment to analyze foreign keys in a specific database/schema
# analyze_foreign_keys("ADVENTUREWORKS", "PUBLIC")

## EDA Insights and Findings

This notebook has automatically explored the structure of your Snowflake database, providing insights into:

1. **Database Overview**: Number and size of available databases 
2. **Schema Analysis**: Structure and organization of schemas within databases
3. **Table Analysis**: Distribution and size of tables across schemas
4. **Column Analysis**: Data types and relationships between columns
5. **Data Exploration**: Sample data from key tables

### Key Observations:

- The largest database explored was [database name] with [size] MB
- A total of [number] tables were analyzed across [number] schemas
- The most common data type was [type] (used in [percentage]% of columns)
- [Number] potential relationships were identified between tables

### Next Steps:

1. Uncomment the deep_dive_database() function call to explore a specific database in more detail
2. Uncomment the analyze_foreign_keys() function to investigate table relationships in a specific schema
3. Modify the exploration parameters to examine more databases or tables
4. Run specific SQL queries on tables of interest using the explorer.cursor.execute() method

This automated EDA provides a solid foundation for understanding your Snowflake database structure and planning further analysis.